# UHVDB r6 duplicate sequence analyses

Analyses for the response to reviewers, using UHVDB r6 metadata and Polars.

1. Duplicate hash groups vs total sequences vs unique sequences
2. Shared sequence hashes between VIRE and SPIRE
3. CF_METAG sequences whose hash appears in releases r1–r4

**Environment:** `lgonsa` micromamba environment

In [1]:
from pathlib import Path

import polars as pl

METADATA_PATH = Path(
    "../../toolkit2/databases/uhvdb/v6/analyze/uhvdb_metadata.tsv.gz"
).resolve()

COLS = ["seq_name", "source_db", "hash", "added_in_release"]

df = pl.read_csv(
    METADATA_PATH,
    separator="\t",
    columns=COLS,
)

print(f"Loaded {df.height:,} rows from {METADATA_PATH}")
df.head()

Loaded 1,541,381 rows from /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/toolkit2/databases/uhvdb/v6/analyze/uhvdb_metadata.tsv.gz


seq_name,source_db,hash,added_in_release
str,str,str,str
"""UHGV-0222920""","""UHGV""","""0f2d77ba3b2e8e1d1ec06023aef50f…","""r1"""
"""UHGV-1519133""","""UHGV""","""ea2deb4ab1f92e9c3fe9049ad88c32…","""r1"""
"""UHGV-0038949""","""UHGV""","""8f781180c6172c31561ebcca7ee049…","""r1"""
"""UHGV-2169955""","""UHGV""","""921d5501a17bc2f115bc92db6a8c27…","""r1"""
"""UHGV-0198886""","""UHGV""","""01685f4201d041399dc6e6def3117d…","""r1"""


## 1. Duplicated sequences in UHVDB r6

- **Total sequences:** number of rows
- **Unique sequences:** number of distinct `hash` values
- **Duplicated sequences:** number of unique hashes that appear ≥2 times

In [2]:
hash_counts = df.group_by("hash").len().rename({"len": "n_copies"})

n_total = df.height
n_unique = hash_counts.height
n_duplicated_hashes = hash_counts.filter(pl.col("n_copies") >= 2).height

summary_1 = pl.DataFrame(
    {
        "metric": [
            "total_sequences",
            "unique_sequences",
            "duplicated_hashes",
        ],
        "count": [n_total, n_unique, n_duplicated_hashes],
    }
)

summary_1

metric,count
str,i64
"""total_sequences""",1541381
"""unique_sequences""",1155490
"""duplicated_hashes""",258735


## 2. Shared hashes between VIRE and SPIRE

- **Shared hashes:** unique `hash` values present in both `source_db == "VIRE"` and `source_db == "SPIRE"`
- **SPIRE sequences sharing a hash with VIRE:** SPIRE rows whose `hash` also appears in VIRE

In [3]:
vire = df.filter(pl.col("source_db") == "VIRE")
spire = df.filter(pl.col("source_db") == "SPIRE")

vire_hashes = vire.select("hash").unique()
spire_hashes = spire.select("hash").unique()
shared_vire_spire = vire_hashes.join(spire_hashes, on="hash", how="inner")

spire_rows_sharing_vire = spire.join(vire_hashes, on="hash", how="inner")

summary_2 = pl.DataFrame(
    {
        "metric": [
            "vire_unique_hashes",
            "spire_unique_hashes",
            "shared_hashes_vire_spire",
            "spire_sequences_sharing_hash_with_vire",
        ],
        "count": [
            vire_hashes.height,
            spire_hashes.height,
            shared_vire_spire.height,
            spire_rows_sharing_vire.height,
        ],
    }
)

summary_2

metric,count
str,i64
"""vire_unique_hashes""",433678
"""spire_unique_hashes""",126391
"""shared_hashes_vire_spire""",88897
"""spire_sequences_sharing_hash_w…",94154


## 3. CF_METAG sequences sharing a hash with releases r1–r4

- **Total CF_METAG sequences:** number of rows with `source_db == "CF_METAG"`
- **CF_METAG sequences sharing a hash with r1–r4:** CF_METAG rows whose `hash` appears among any sequence with `added_in_release` in `{r1, r2, r3, r4}`

In [4]:
cf_metag = df.filter(pl.col("source_db") == "CF_METAG")
r1_r4_hashes = (
    df.filter(pl.col("added_in_release").is_in(["r1", "r2", "r3", "r4"]))
    .select("hash")
    .unique()
)

cf_overlap_rows = cf_metag.join(r1_r4_hashes, on="hash", how="inner")

summary_3 = pl.DataFrame(
    {
        "metric": [
            "total_cf_metag_sequences",
            "cf_metag_unique_hashes",
            "r1_r4_unique_hashes",
            "cf_metag_rows_sharing_hash_with_r1_r4",
        ],
        "count": [
            cf_metag.height,
            cf_metag.select("hash").n_unique(),
            r1_r4_hashes.height,
            cf_overlap_rows.height,
        ],
    }
)

summary_3

metric,count
str,i64
"""total_cf_metag_sequences""",4732
"""cf_metag_unique_hashes""",4202
"""r1_r4_unique_hashes""",575287
"""cf_metag_rows_sharing_hash_wit…",72
